# SaRoHead - BERT with intermediate task transfer learning

The MLM backbones (`job_TTL.sh`) must be on Drive under `MyDrive/sarohead/unsupervised_TTL/<task>/backup_<task>_<BERT|Distil|XLM>/`.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/sarohead")
REPO = Path("/content/SaRoHead")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/GRAI-UNSTPB/SaRoHead.git", str(REPO)], check=True)

RESULTS = DRIVE / "results" / "predictions"
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["SAROHEAD_RESULTS"] = str(RESULTS)
sys.path.insert(0, str(REPO))
from analysis_scripts.prediction_io import prediction_filename

SEEDS = [13, 42, 123, 2024, 7]
CATEGORIES = ["social", "politic", "sport"]


def run(cmd, cwd):
    print(">>", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd), check=True)


def done(method, model, variant, category, seed):
    return (RESULTS / method / prediction_filename(method, model, variant, category, seed, "random", "test")).exists()


In [ ]:
!pip install -q lightning transformers pandas scikit-learn

MODELS = ["bert", "distilled-bert", "xlm-roberta"]
INTER_TASKS = ["standard", "click", "saroco", "scitechbait"]
SHORT = {"bert": "BERT", "distilled-bert": "Distil", "xlm-roberta": "XLM"}
LABEL = {"standard": "None", "click": "RoCliCo", "saroco": "SaRoCo", "scitechbait": "SciTechBaitRo"}

for task in INTER_TASKS[1:]:
    for model in MODELS:
        name = f"backup_{task}_{SHORT[model]}"
        dst = REPO / "unsupervised_TTL" / task / name
        if not dst.exists():
            shutil.copytree(DRIVE / "unsupervised_TTL" / task / name, dst)


In [ ]:
for seed in SEEDS:
    for model in MODELS:
        for task in INTER_TASKS:
            cats = [c for c in CATEGORIES if not done("bert", model, LABEL[task], c, seed)]
            if not cats:
                continue
            run([sys.executable, "train_BERT_satire.py", "--model_type", model, "--lr", "1e-3", "--sch_type", "linear",
                 "--inter_task", task, "--num_epochs", 35, "--seed", seed, "--categories", *cats], REPO / "bert_scripts")
            shutil.rmtree(REPO / "bert_scripts" / "checkpoints", ignore_errors=True)


In [ ]:
files = sorted(RESULTS.rglob("*.csv"))
print(len(files), "prediction files in", RESULTS)
